# AIC 2026 — Query encoder server · SigLIP 2 + BEiT-3

Notebook chạy trên **Google Colab** (hoặc **Kaggle**, cần bật Internet) để phục vụ web UI
tìm kiếm chạy trên máy local: máy local gửi câu truy vấn, notebook trả về vector của
SigLIP 2 và BEiT-3 bằng đúng model và revision đã dùng khi ingest ảnh. Máy local không
cần tải model.

Cách mã hoá giống hệt `encode-query-vectors.ipynb` (text tiếng Anh; SigLIP 2 viết thường,
pad 64 token; BEiT-3 giữ nguyên hoa/thường, tối đa 64 token), nên vector trả về trùng với
vector của notebook đó.

**Input:** `manifest.json` lấy từ `indexes/dev/manifest.json` trên máy local.

**Chạy:** Runtime → Run all. Ô cuối in ra `ENCODER_URL` và `ENCODER_KEY`; trên máy local:

```powershell
$env:ENCODER_KEY = "<key in ra>"
uv run python scripts/serve_ui.py --encoder-url <ENCODER_URL>
```

Tunnel dùng Cloudflare quick tunnel (không cần tài khoản), địa chỉ đổi mỗi lần chạy.
`ENCODER_KEY` sinh ngẫu nhiên mỗi phiên và chỉ in ra màn hình: không lưu vào notebook,
không chia sẻ. Xoá output trước khi lưu notebook. Server dừng khi runtime tắt.

In [ ]:
# transformers 4.57.1 matches the SigLIP 2 ingest notebook; timm 0.9.16 matches BEiT-3.
import importlib.metadata
import subprocess
import sys


def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


REQUIRED = {"transformers": "4.57.1", "timm": "0.9.16"}
missing = [f"{name}=={version}" for name, version in REQUIRED.items() if installed_version(name) != version]
for name in ("sentencepiece", "fastapi", "uvicorn"):
    if installed_version(name) is None:
        missing.append(name)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print({name: installed_version(name) for name in ("torch", "transformers", "timm", "fastapi", "uvicorn")})

In [ ]:
import gc
import hashlib
import json
import os
import sys
import tarfile
import types
from pathlib import Path

import numpy as np
import requests
import torch

MANIFEST_JSON = "manifest.json"
ENCODE = ("siglip", "beit3")
PORT = 8765
MAX_TEXTS = 64  # per request
MAX_CHARS = 2000  # per text; both encoders truncate at 64 tokens anyway
SERVER_VERSION = 1

SIGLIP_MAX_LENGTH = 64
BEIT3_MAX_LENGTH = 64
BEIT3_SOURCE_REVISION = "ca43e4cd19445a536f133bf2bc25b573b2f0c7c5"
BEIT3_SOURCE_SHA256 = {
    "modeling_utils.py": "cc04b762b2c0ba32eb82d65a5e543adada156ce3584a351b56c87eb3de293a27",
    "modeling_finetune.py": "dfa9a2dbba7e7b2a46023da0d693f6e781d946d86b2b7a52e69ed4bd463d763f",
}
BEIT3_CHECKPOINT_URL = (
    "https://github.com/addf400/files/releases/download/beit3/"
    "beit3_large_patch16_384_coco_retrieval.pth"
)
BEIT3_CHECKPOINT_SIZE = 1_350_590_595
TORCHSCALE_URL = (
    "https://files.pythonhosted.org/packages/d5/7f/"
    "2a20aeaf3264eb41750b882fc56a67d907042dcf3f84745331ee4e6e7636/torchscale-0.2.0.tar.gz"
)
TORCHSCALE_SHA256 = "ca0a3982d799d6b91320a538a6cc17ea9ccb6370231934805fae78f318fe4d2f"
BEIT3_TOKENIZER_URL = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"


def runtime_name():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


RUNTIME = runtime_name()
WORK = (
    Path("/content") if RUNTIME == "colab"
    else Path("/kaggle/working") if RUNTIME == "kaggle"
    else Path.cwd()
)
MODEL_ROOT = WORK / "query-encoders"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)


def find_manifest():
    candidates = [WORK / MANIFEST_JSON, Path.cwd() / MANIFEST_JSON]
    if RUNTIME == "kaggle":
        candidates += sorted(Path("/kaggle/input").rglob(MANIFEST_JSON))
    for path in candidates:
        if path.is_file():
            return path
    if RUNTIME == "colab":
        from google.colab import files

        print("Upload indexes/dev/manifest.json")
        uploaded = [Path(key) for key in files.upload()]
        picked = [path for path in uploaded if path.suffix == ".json"]
        if len(picked) == 1:
            return picked[0].resolve()
    raise FileNotFoundError(f"{MANIFEST_JSON} not found; upload it to {WORK}")


manifest = json.loads(find_manifest().read_text(encoding="utf-8"))
MODELS = {}
for module in ENCODE:
    entry = manifest["faiss"][module]
    if len(entry.get("model_revisions", [])) != 1:
        raise ValueError(f"{module}: manifest must record exactly one ingest revision")
    MODELS[module] = {
        "model_id": entry["model_ids"][0],
        "model_revision": entry["model_revisions"][0],
        "dimension": entry["dimension"],
    }
print("Runtime:", RUNTIME, "| device:", DEVICE)
print(json.dumps(MODELS, indent=2))

## Nạp hai text encoder

Hàm mã hoá lấy nguyên văn từ `encode-query-vectors.ipynb`; ở đây model được nạp một lần
và giữ trong bộ nhớ, thay vì nạp lại cho mỗi lô câu.

In [ ]:
def encode_siglip(texts):
    from transformers import AutoTokenizer, Siglip2TextModel

    spec = MODELS["siglip"]
    tokenizer = AutoTokenizer.from_pretrained(spec["model_id"], revision=spec["model_revision"])
    model = Siglip2TextModel.from_pretrained(
        spec["model_id"], revision=spec["model_revision"], torch_dtype=torch.float32
    ).to(DEVICE).eval()
    outputs = []
    for start in range(0, len(texts), 32):
        batch = [" ".join(str(text).lower().split()) for text in texts[start:start + 32]]
        inputs = tokenizer(
            batch, padding="max_length", max_length=SIGLIP_MAX_LENGTH,
            truncation=True, return_tensors="pt",
        )
        with torch.inference_mode():
            features = model(input_ids=inputs["input_ids"].to(DEVICE)).pooler_output
        outputs.append(torch.nn.functional.normalize(features.float(), dim=-1).cpu().numpy())
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(outputs)

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def download(url, destination, expected_size=None):
    destination = Path(destination)
    if destination.exists() and (expected_size is None or destination.stat().st_size == expected_size):
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    print("Downloading", url)
    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for block in response.iter_content(chunk_size=8 * 1024 * 1024):
                stream.write(block)
    if expected_size is not None and partial.stat().st_size != expected_size:
        raise RuntimeError(f"{url}: unexpected size {partial.stat().st_size}")
    partial.replace(destination)
    return destination


def verified(path, expected_sha256):
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise RuntimeError(f"{Path(path).name}: SHA-256 {actual} != {expected_sha256}")
    return Path(path)


def beit3_text_inputs(token_ids, bos, eos, pad, max_length=BEIT3_MAX_LENGTH):
    """Mirror UniLM BaseDataset._get_text_segment: bos + ids + eos, pad; mask 1 on padding."""
    if not token_ids:
        raise ValueError("BEiT-3 query must contain at least one token")
    tokens = [bos, *token_ids[: max_length - 2], eos]
    padding = max_length - len(tokens)
    return tokens + [pad] * padding, [0] * len(tokens) + [1] * padding


def load_beit3():
    from transformers import XLMRobertaTokenizer

    root = MODEL_ROOT / "beit3"
    source_dir = root / "unilm"
    for filename, expected in BEIT3_SOURCE_SHA256.items():
        url = (
            "https://raw.githubusercontent.com/microsoft/unilm/"
            f"{BEIT3_SOURCE_REVISION}/beit3/{filename}"
        )
        verified(download(url, source_dir / filename), expected)
    checkpoint = verified(
        download(
            BEIT3_CHECKPOINT_URL,
            root / "beit3_large_patch16_384_coco_retrieval.pth",
            expected_size=BEIT3_CHECKPOINT_SIZE,
        ),
        MODELS["beit3"]["model_revision"],
    )
    archive = verified(download(TORCHSCALE_URL, root / "torchscale-0.2.0.tar.gz"), TORCHSCALE_SHA256)
    torchscale_root = root / "torchscale-0.2.0"
    if not (torchscale_root / "torchscale" / "__init__.py").exists():
        with tarfile.open(archive, mode="r:gz") as tar:
            tar.extractall(root, filter="data")
    spm = download(BEIT3_TOKENIZER_URL, root / "beit3.spm")

    fairscale = types.ModuleType("fairscale")
    fairscale_nn = types.ModuleType("fairscale.nn")
    fairscale_nn.checkpoint_wrapper = lambda module, *args, **kwargs: module
    fairscale_nn.wrap = lambda module, *args, **kwargs: module
    fairscale.nn = fairscale_nn

    class InferenceOnlyClipLoss(torch.nn.Module):
        def forward(self, *args, **kwargs):
            raise RuntimeError("Training loss is unavailable at query time")

    utils_shim = types.ModuleType("utils")
    utils_shim.ClipLoss = lambda *args, **kwargs: InferenceOnlyClipLoss()
    utils_shim.get_rank = lambda: 0
    utils_shim.get_world_size = lambda: 1
    sys.modules.update({"fairscale": fairscale, "fairscale.nn": fairscale_nn, "utils": utils_shim})
    sys.path[:0] = [str(torchscale_root), str(source_dir)]
    from modeling_finetune import beit3_large_patch16_384_retrieval

    model = beit3_large_patch16_384_retrieval(pretrained=False)
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model.load_state_dict(state.get("model", state), strict=True)
    del state
    return model.to(DEVICE).eval(), XLMRobertaTokenizer(str(spm))


def encode_beit3(texts):
    model, tokenizer = load_beit3()
    outputs = []
    for start in range(0, len(texts), 32):
        batch_tokens, batch_padding = [], []
        for text in texts[start:start + 32]:
            ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(" ".join(str(text).split())))
            tokens, padding = beit3_text_inputs(
                ids, tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id
            )
            batch_tokens.append(tokens)
            batch_padding.append(padding)
        with torch.inference_mode():
            _, language = model(
                text_description=torch.tensor(batch_tokens, device=DEVICE),
                padding_mask=torch.tensor(batch_padding, device=DEVICE),
                only_infer=True,
            )
        outputs.append(torch.nn.functional.normalize(language.float(), dim=-1).cpu().numpy())
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(outputs)

In [ ]:
import threading
from transformers import AutoTokenizer, Siglip2TextModel

_spec = MODELS["siglip"]
SIGLIP_TOKENIZER = AutoTokenizer.from_pretrained(_spec["model_id"], revision=_spec["model_revision"])
SIGLIP_MODEL = Siglip2TextModel.from_pretrained(
    _spec["model_id"], revision=_spec["model_revision"], torch_dtype=torch.float32
).to(DEVICE).eval()
BEIT3_MODEL, BEIT3_TOKENIZER = load_beit3()
LOCK = threading.Lock()


def siglip_vectors(texts):
    inputs = SIGLIP_TOKENIZER(
        [" ".join(str(text).lower().split()) for text in texts],
        padding="max_length", max_length=SIGLIP_MAX_LENGTH, truncation=True, return_tensors="pt",
    )
    with torch.inference_mode():
        features = SIGLIP_MODEL(input_ids=inputs["input_ids"].to(DEVICE)).pooler_output
    return torch.nn.functional.normalize(features.float(), dim=-1).cpu().numpy()


def beit3_vectors(texts):
    batch_tokens, batch_padding = [], []
    for text in texts:
        ids = BEIT3_TOKENIZER.convert_tokens_to_ids(BEIT3_TOKENIZER.tokenize(" ".join(str(text).split())))
        tokens, padding = beit3_text_inputs(
            ids, BEIT3_TOKENIZER.bos_token_id, BEIT3_TOKENIZER.eos_token_id, BEIT3_TOKENIZER.pad_token_id
        )
        batch_tokens.append(tokens)
        batch_padding.append(padding)
    with torch.inference_mode():
        _, language = BEIT3_MODEL(
            text_description=torch.tensor(batch_tokens, device=DEVICE),
            padding_mask=torch.tensor(batch_padding, device=DEVICE),
            only_infer=True,
        )
    return torch.nn.functional.normalize(language.float(), dim=-1).cpu().numpy()


# Same vectors as the batch notebook's functions (which reload the models per call).
_probe = ["A man cooking shrimp in a pan.", "Two news anchors in a studio"]
assert np.allclose(siglip_vectors(_probe), encode_siglip(_probe), atol=1e-5)
assert np.allclose(beit3_vectors(_probe), encode_beit3(_probe), atol=1e-5)
print("Encoders loaded and checked against the batch notebook functions.")

## Mở API qua Cloudflare quick tunnel

`POST /encode` nhận `{"texts": [...]}` và trả về vector hai encoder; `GET /health` trả về
model/revision để máy local đối chiếu với index. Mọi request phải có header `X-API-Key`.

In [ ]:
import re
import secrets
import time

import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

ENCODER_KEY = secrets.token_urlsafe(24)
META = {"server_version": SERVER_VERSION, "device": DEVICE, "models": MODELS}
app = FastAPI()


class EncodeRequest(BaseModel):
    texts: list[str]


def authorize(key):
    if not secrets.compare_digest(key or "", ENCODER_KEY):
        raise HTTPException(status_code=401, detail="invalid X-API-Key")


@app.get("/health")
def health(x_api_key: str | None = Header(default=None)):
    authorize(x_api_key)
    return META


@app.post("/encode")
def encode(request: EncodeRequest, x_api_key: str | None = Header(default=None)):
    authorize(x_api_key)
    texts = [" ".join(text.split()) for text in request.texts]
    if not texts or len(texts) > MAX_TEXTS:
        raise HTTPException(status_code=400, detail=f"send 1..{MAX_TEXTS} texts")
    if any(not text or len(text) > MAX_CHARS for text in texts):
        raise HTTPException(status_code=400, detail=f"texts must be 1..{MAX_CHARS} characters")
    with LOCK:
        vectors = {"siglip": siglip_vectors(texts), "beit3": beit3_vectors(texts)}
    return {**META, **{module: matrix.tolist() for module, matrix in vectors.items()}}


server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(50):
    if server.started:
        break
    time.sleep(0.2)
if not server.started:
    raise RuntimeError("uvicorn did not start")

CLOUDFLARED = MODEL_ROOT / "cloudflared"
if not CLOUDFLARED.exists():
    subprocess.check_call([
        "wget", "-q", "-O", str(CLOUDFLARED),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ])
    CLOUDFLARED.chmod(0o755)
tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
ENCODER_URL = None
deadline = time.time() + 60
while ENCODER_URL is None and time.time() < deadline:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        ENCODER_URL = match.group(0)
if ENCODER_URL is None:
    raise RuntimeError("cloudflared did not report a tunnel URL")
# Keep draining cloudflared's log so its pipe never fills up.
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print("ENCODER_URL =", ENCODER_URL)
print("ENCODER_KEY =", ENCODER_KEY)
print("\nOn the local machine (PowerShell):")
print(f'  $env:ENCODER_KEY = "{ENCODER_KEY}"')
print(f"  uv run python scripts/serve_ui.py --encoder-url {ENCODER_URL}")
print("\nKeep this runtime open while searching. The tunnel may need ~10 s before it answers.")